# Разбор упражнений к лекции 10

Условия — раздел 9 конспекта [`lecture10.md`](lecture10.md); краткие ответы без кода — [`exercises10.md`](exercises10.md). Все упражнения — домашние; семинар — у доски, задачи в §8 конспекта. **Сначала попробуйте сами**, потом сверяйтесь. Каждый разбор: условие → решение по шагам → проверка кодом там, где она помогает. Нужен CasADi: `uv run --group casadi jupyter lab`; в Colab первая ячейка ставит его сама.

In [1]:
%matplotlib inline

import importlib.util, subprocess, sys
if importlib.util.find_spec("casadi") is None:          # Colab: поставить CasADi один раз
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "casadi"], check=True)

import casadi as ca
import numpy as np
import matplotlib.pyplot as plt
from l10helpers import *  # задачи «парк» и «портфель» (собираются из build_demo10.py)

## 10.0. Разминка — счёт руками

**Условие.** *Суть: одно ограничение в двух случаях — активно и неактивно.*

Задача: $\min_{x\in\mathbb R^2}\ x_1^2+x_2^2$ при $x_1+x_2-c\ge0$.

- (а) Выпишите условия ККТ.
- (б) Решите при $c=2$ и при $c=-2$. Активно ли ограничение и чему равен $\mu$?

**Решение.**

**Шаг 1. Условия ККТ.** Здесь $f=x_1^2+x_2^2$, $h=x_1+x_2-c$, $\nabla f=(2x_1,\ 2x_2)$, $\nabla h=(1,\ 1)$:

$$
2x_1=\mu,\qquad 2x_2=\mu,\qquad x_1+x_2-c\ge0,\qquad \mu\ge0,\qquad \mu\,(x_1+x_2-c)=0 .
$$

**Шаг 2. Симметрия.** Из первых двух уравнений $x_1=x_2=\mu/2$.

**Шаг 3. $c=2$.** Пусть $\mu=0$. Тогда $x=(0,\ 0)$ и $h=-2<0$ — недопустимо. Значит, $\mu>0$ и ограничение активно:

$$
x_1+x_2=\mu=2,\qquad x^\ast=(1,\ 1),\qquad \mu=2\ge0 .
$$

**Шаг 4. $c=-2$.** Пусть $\mu=0$. Тогда $x=(0,\ 0)$ и $h=0+0+2=2>0$ — допустимо. Ограничение неактивно, $\mu=0$, $x^\ast=(0,\ 0)$.

## 10.1. Задача «парк» ещё раз

**Условие.** *Суть: на границе может быть и $\mu=0$ — тогда равны нулю оба числа.*

Задача: $\min_{x\in\mathbb R^2}\ \tfrac12\lVert x-a\rVert^2$ при $1-x_1^2-x_2^2\ge0$.

- (а) Дом $a=(0,\ 3)$. Найдите $x^\ast$ и $\mu$.
- (б) Дом на границе, $a=(0.6,\ 0.8)$. Чему равны $\mu$ и $h$ в ответе? Почему $\mu h=0$ выполнено и здесь?

**Решение.**

**Шаг 1. Формула ответа.** Из стационарности $x-a=-2\mu x$ следует $x=a/(1+2\mu)$ — точка на отрезке от центра к дому. Если $\lVert a\rVert\le1$, ответ — сам дом и $\mu=0$. Иначе $\lVert x\rVert=1$ даёт $\mu=(\lVert a\rVert-1)/2$.

**Шаг 2. (а) Дом $a=(0,\ 3)$.** $\lVert a\rVert=3$, поэтому

$$
\mu=\frac{3-1}{2}=1,\qquad x^\ast=\frac{a}{\lVert a\rVert}=(0,\ 1) .
$$

**Шаг 3. (б) Дом $a=(0.6,\ 0.8)$.** $\lVert a\rVert=1$ — дом на границе. Ответ — сам дом, $x^\ast=a$. Тогда $h=1-0.36-0.64=0$ и $x-a=0$, откуда $\mu=0$. Условие $\mu h=0$ выполнено: равны нулю оба числа. Ограничение активно, но его множитель равен нулю.

In [2]:
for a in ((0.0, 3.0), (0.6, 0.8)):
    x, mu = park_solution(a)                                   # формула ответа из §2.1
    h = 1 - x @ x
    print(f"a = {a}: x* = {x.round(4)}, mu = {mu:.4f}, h = {h:.1e}, mu*h = {mu * h:.1e}")

a = (0.0, 3.0): x* = [0. 1.], mu = 1.0000, h = 0.0e+00, mu*h = 0.0e+00
a = (0.6, 0.8): x* = [0.6 0.8], mu = 0.0000, h = 0.0e+00, mu*h = 0.0e+00


**Что видим.** Для дома $(0,\ 3)$: $x^\ast=(0,\ 1)$, $\mu=1$. Для дома на границе $h=0$ и $\mu=0$ одновременно.

## 10.2. Квадрат: два активных ограничения

**Условие.** (с кодом). *Суть: в углу у ответа два множителя, и оба неотрицательны.*

Задача: $\min_{x\in\mathbb R^2}\ \tfrac12\lVert x-a\rVert^2$ при $0\le x_1\le1$ и $0\le x_2\le1$ — четыре неравенства.

- (а) Для $a=(2,\ 2)$ и $a=(2,\ 0.5)$ найдите руками ответ, активное множество и множители.
- (б) Проверьте IPOPT: границы в `lbx` и `ubx`, множители в `lam_x`. Какой у них знак при активной верхней границе?

**Решение.**

**Шаг 1. Ограничения.** Четыре неравенства: $h_1=x_1\ge0$, $h_2=x_2\ge0$, $h_3=1-x_1\ge0$, $h_4=1-x_2\ge0$. Градиенты: $e_1$, $e_2$, $-e_1$, $-e_2$.

**Шаг 2. $a=(2,\ 2)$.** Угадаем угол $(1,\ 1)$: активны $h_3$ и $h_4$. Стационарность:

$$
x-a=(-1,\ -1)=\mu_3(-1,\ 0)+\mu_4(0,\ -1)\quad\Longrightarrow\quad \mu_3=\mu_4=1\ge0 .
$$

Обе проверки проходят: $x_1=x_2=1\ge0$, оба множителя неотрицательны. Ответ $x^\ast=(1,\ 1)$, $\mathcal A=\{3,\ 4\}$.

**Шаг 3. $a=(2,\ 0.5)$.** Угадаем $\mathcal A=\{3\}$: $x_1=1$, вторая координата свободна. Стационарность по $x_2$: $x_2-0.5=0$, то есть $x_2=0.5$. По $x_1$: $1-2=-\mu_3$, $\mu_3=1\ge0$. Ответ $x^\ast=(1,\ 0.5)$.

**Шаг 4. (б) IPOPT.** Ячейка ниже решает обе задачи с границами `lbx=0`, `ubx=1`.

In [3]:
x = ca.SX.sym("x", 2)
for a in ((2.0, 2.0), (2.0, 0.5)):
    nlp = {"x": x, "f": 0.5 * ca.sumsqr(x - ca.DM(a))}                  # половина квадрата расстояния до a
    solver = ca.nlpsol("box", "ipopt", nlp, {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
    res = solver(x0=[0.5, 0.5], lbx=0, ubx=1)                           # квадрат [0, 1]^2 — границы переменных
    print(f"a = {a}: x* = {np.array(res['x']).ravel().round(4)}, lam_x = {np.array(res['lam_x']).ravel().round(4)}")

a = (2.0, 2.0): x* = [1. 1.], lam_x = [1. 1.]
a = (2.0, 0.5): x* = [1.  0.5], lam_x = [1. 0.]


**Что видим.** У активной верхней границы `lam_x` $=+1$ — это наш $\mu$ с плюсом. У нижней границы в портфеле `lam_x` был отрицателен. Правило: знак `lam_x` говорит, какая граница активна, модуль — наш множитель.

## 10.3. Портфель при других доходностях

**Условие.** (с кодом). *Суть: активное множество меняется с требуемой доходностью.*

Задача портфеля из введения при $\rho=8\,\%$ и $\rho=14\,\%$. Функция `portfolio_active(rho, A)` из `l10helpers.py` решает систему для заданного активного множества `A`.

- (а) Найдите активное множество алгоритмом «угадай и проверь», начав с $\mathcal A=\varnothing$. Сколько догадок понадобилось?
- (б) Сравните цену доходности $\lambda_\rho$ с запретом коротких продаж и без него.

**Решение.**

**Шаг 1. Алгоритм.** Начинаем с $\mathcal A=\varnothing$. Если какая-то доля отрицательна, добавляем в $\mathcal A$ самую отрицательную. Если какой-то $\mu_i<0$, убираем это ограничение из $\mathcal A$. Ячейка ниже делает это и печатает все догадки.

In [4]:
def guess_and_check(rho, maxit=10):
    A = []                                                              # стартовая догадка: ничего не активно
    for k in range(1, maxit + 1):
        x, lam1, lam_rho, mu = portfolio_active(rho, A)                 # система лекции 8 с x_i = 0 для i из A
        print(f"   догадка {k}: A = {[i + 1 for i in A]}, x = {x.round(4)}, lam_rho = {lam_rho:.4f}, mu = {mu.round(4)}")
        neg = [i for i in range(4) if i not in A and x[i] < -1e-12]     # нарушенные x_i >= 0
        if neg:
            A = sorted(A + [min(neg, key=lambda i: x[i])])              # добавить самую отрицательную долю
            continue
        bad = [i for i, m in zip(A, mu) if m < -1e-12]                  # отрицательные множители
        if bad:
            A.remove(bad[0])
            continue
        return A, lam_rho

for rho in (0.08, 0.14):
    print(f"rho = {rho:.0%}:")
    A, lam_rho = guess_and_check(rho)
    lam_free = portfolio_active(rho, [])[2]                             # цена доходности без запрета
    print(f"   ответ: A = {[i + 1 for i in A]}; lam_rho с запретом {lam_rho:.4f}, без запрета {lam_free:.4f}")

rho = 8%:
   догадка 1: A = [], x = [0.3908 0.2662 0.2095 0.1336], lam_rho = 0.1630, mu = []
   ответ: A = []; lam_rho с запретом 0.1630, без запрета 0.1630
rho = 14%:
   догадка 1: A = [], x = [-0.4977  0.6141  0.5439  0.3397], lam_rho = 0.4452, mu = []
   догадка 2: A = [1], x = [ 0.     -0.267   0.7173  0.5497], lam_rho = 0.7464, mu = [0.0203]
   догадка 3: A = [1, 2], x = [ 0.   -0.    0.25  0.75], lam_rho = 1.2150, mu = [0.061 0.023]
   ответ: A = [1, 2]; lam_rho с запретом 1.2150, без запрета 0.4452


**Шаг 2. $\rho=8\,\%$.** Первая догадка $\mathcal A=\varnothing$ проходит: все доли положительны. Ограничения $x\ge0$ не работают, и $\lambda_\rho=0.163$ — как без запрета.

**Шаг 3. $\rho=14\,\%$.** Нужны три догадки:

$$
\varnothing\ \xrightarrow{\ x_1=-0.50\ }\ \{1\}\ \xrightarrow{\ x_2=-0.27\ }\ \{1,\ 2\} .
$$

Ответ $x=(0,\ 0,\ 0.25,\ 0.75)$, $\mu=(0.061,\ 0.023)$. Цена доходности $\lambda_\rho=1.215$ против $0.445$ без запрета — почти втрое выше.

## 10.4. Для желающих: когда множителей нет

**Условие.** *Суть: без LICQ в минимуме условия ККТ могут не выполняться.*

Задача:

$$
\min_{x\in\mathbb R^2}\ -x_1\qquad\text{при}\qquad h_1(x)=(1-x_1)^3-x_2\ge0,\quad h_2(x)=x_2\ge0 .
$$

- (а) Докажите, что решение — точка $(1,\ 0)$.
- (б) Покажите, что в ней нет $\mu_1$, $\mu_2$ с $\nabla f=\mu_1\nabla h_1+\mu_2\nabla h_2$. Какое условие теоремы ККТ нарушено?

**Решение.**

**Шаг 1. (а) Решение.** Из двух ограничений $0\le x_2\le(1-x_1)^3$. Это возможно, только если $(1-x_1)^3\ge0$, то есть $x_1\le1$. Минимум $-x_1$ — при наибольшем $x_1$:

$$
x_1=1,\qquad 0\le x_2\le0\quad\Longrightarrow\quad x^\ast=(1,\ 0) .
$$

**Шаг 2. (б) Градиенты в $(1,\ 0)$.** Оба ограничения активны:

$$
\nabla f=(-1,\ 0),\qquad \nabla h_1=\big(-3(1-x_1)^2,\ -1\big)=(0,\ -1),\qquad \nabla h_2=(0,\ 1) .
$$

**Шаг 3. Множителей нет.** Условие $\nabla f=\mu_1\nabla h_1+\mu_2\nabla h_2$ покомпонентно:

$$
-1=\mu_1\cdot0+\mu_2\cdot0,\qquad 0=-\mu_1+\mu_2 .
$$

Первое равенство $-1=0$ не выполняется ни при каких $\mu$.

**Шаг 4. Что нарушено.** LICQ: $\nabla h_1=-\nabla h_2$, градиенты активных ограничений линейно зависимы. Теорема ККТ к этой точке не применима.

In [5]:
x = ca.SX.sym("x", 2)
nlp = {"x": x, "f": -x[0], "g": ca.vertcat((1 - x[0]) ** 3 - x[1], x[1])}   # оба неравенства в g >= 0
solver = ca.nlpsol("kt", "ipopt", nlp, {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
res = solver(x0=[0.5, 0.05], lbg=0, ubg=ca.inf)
st = solver.stats()
print("статус:", st["return_status"], "| итераций:", st["iter_count"])
print("x =", np.array(res["x"]).ravel().round(4), "| lam_g =", np.array(res["lam_g"]).ravel().round(0))

статус: Solve_Succeeded | итераций: 24
x = [ 1.0027 -0.    ] | lam_g = [-45240. -45240.]


**Что видим.** IPOPT сообщает об успехе и останавливается рядом с $(1,\ 0)$. Но множители огромны — больше $10^4$, а $x_1\approx1.003$ вместо $1$. Солвер пытается выполнить условие, которое в этой точке не выполняется. Огромные множители — признак нарушенного LICQ, как в лекции 8.